# Analyzing Positional Advantage from Qualifying Races

## Sprint Race Impact

**How Sprint Races Work**

- Introduced        : 2021 season
- Race Day          : Usually on Saturday
- Length            : ~100 km (about 25–30 minutes)
- No Pit Stops      : Drivers usually complete it on a single tire set
- Purpose           : Orignally used to determines the starting grid for Sunday’s main race (earlier), or standalone (recent)

**Evolution**

- 2021	: Introduced at 3 races. Sprint result decided Sunday grid. Points: 1st–3rd only (3–2–1)
- 2022	: Increased to 3 races. Points extended to top 8 (8–7–6...1). Sprint still affected Sunday grid.
- 2023	: Expanded to 6 sprint races. Sprint became standalone, no longer decided Sunday grid. Sprint shootout (a shorter qualifying) decided sprint grid.
- 2024+	: Continues with same standalone format and sprint shootout.

**Key Differences with Sprint Weekends**

- Fewer Practice Sessions: Only one free practice session compared to the usual three.
- Two Qualifying Sessions: One to set the grid for the Sprint, and another to set the grid for the Grand Prix.
- Sprint Race as an Additional Race: A shorter, fast-paced race on Saturday with points awarded to the top 8 finishers (8 points for 1st, down to 1 point for 8th).
- Independent Grids: The Sprint Qualifying sets the Sprint grid, and the Grand Prix Qualifying sets the Grand Prix grid, with no direct link between the two.

## Data Cleaning

### Importing Libraries


In [1]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

### Raw Datasets

In [2]:
drivers_df      = pd.read_csv("../../data/ergast/drivers.csv")
races_df        = pd.read_csv("../../data/ergast/races.csv")
results_df      = pd.read_csv("../../data/ergast/results.csv")
# qualifying_df   = pd.read_csv("../../data/ergast/qualifying.csv") # not used as grid position gives equivalent info
status_df       = pd.read_csv("../../data/ergast/status.csv")

# additional datasets
# sprint_results_df   = pd.read_csv("../../data/ergast/sprint_results.csv")

### Feature Selection

In [3]:
# selecting only relevant columns
drivers_df = drivers_df[["driverId", "code", "surname", "forename"]]
drivers_df.head()

,driverId,code,surname,forename
0,1,HAM,Hamilton,Lewis
1,2,HEI,Heidfeld,Nick
2,3,ROS,Rosberg,Nico
3,4,ALO,Alonso,Fernando
4,5,KOV,Kovalainen,Heikki


In [4]:
races_df = races_df[["raceId", "year", "circuitId", "date", "name", "round"]]
races_df.head()

,raceId,year,circuitId,date,name,round
0,1,2009,1,2009-03-29,Australian Grand Prix,1
1,2,2009,2,2009-04-05,Malaysian Grand Prix,2
2,3,2009,17,2009-04-19,Chinese Grand Prix,3
3,4,2009,3,2009-04-26,Bahrain Grand Prix,4
4,5,2009,4,2009-05-10,Spanish Grand Prix,5


In [5]:
results_df = results_df[["resultId", "raceId", "driverId", "constructorId", "grid", "position", "positionText", "rank", "points", "time", "statusId"]]
results_df.head()

,resultId,raceId,driverId,constructorId,grid,position,positionText,rank,points,time,statusId
0,1,18,1,1,1,1,1,2,10.0,1:34:50.616,1
1,2,18,2,2,5,2,2,3,8.0,+5.478,1
2,3,18,3,3,7,3,3,5,6.0,+8.163,1
3,4,18,4,4,11,4,4,7,5.0,+17.181,1
4,5,18,5,1,3,5,5,1,4.0,+18.014,1


In [6]:
status_df.head()

,statusId,status
0,1,Finished
1,2,Disqualified
2,3,Accident
3,4,Collision
4,5,Engine


### Merging datasets

In [7]:
merged_df = (
    results_df
    .merge(drivers_df, on="driverId", how="left")
    .merge(races_df, on="raceId", how="left")
    .merge(status_df, on="statusId", how="left")
    .set_index("resultId")
)
merged_df.head()

,raceId,driverId,constructorId,grid,position,positionText,rank,points,time,statusId,code,surname,forename,year,circuitId,date,name,round,status
resultId,,,,,,,,,,,,,,,,,,,
1,18,1,1,1,1,1,2,10.0,1:34:50.616,1,HAM,Hamilton,Lewis,2008,1,2008-03-16,Australian Grand Prix,1,Finished
2,18,2,2,5,2,2,3,8.0,+5.478,1,HEI,Heidfeld,Nick,2008,1,2008-03-16,Australian Grand Prix,1,Finished
3,18,3,3,7,3,3,5,6.0,+8.163,1,ROS,Rosberg,Nico,2008,1,2008-03-16,Australian Grand Prix,1,Finished
4,18,4,4,11,4,4,7,5.0,+17.181,1,ALO,Alonso,Fernando,2008,1,2008-03-16,Australian Grand Prix,1,Finished
5,18,5,1,3,5,5,1,4.0,+18.014,1,KOV,Kovalainen,Heikki,2008,1,2008-03-16,Australian Grand Prix,1,Finished


### Data Profiling

In [8]:
merged_df.shape

(26759, 19)

In [9]:
# data types
print("Column Data Types:\n", merged_df.dtypes)

Column Data Types:
 raceId             int64
driverId           int64
constructorId      int64
grid               int64
position          object
positionText      object
rank              object
points           float64
time              object
statusId           int64
code              object
surname           object
forename          object
year               int64
circuitId          int64
date              object
name              object
round              int64
status            object
dtype: object


In [10]:
# missing values
print("\nMissing Values per column:\n", merged_df.isna().sum())


Missing Values per column:
 raceId           0
driverId         0
constructorId    0
grid             0
position         0
positionText     0
rank             0
points           0
time             0
statusId         0
code             0
surname          0
forename         0
year             0
circuitId        0
date             0
name             0
round            0
status           0
dtype: int64


### Standardizing Columns

For drivers, who have not finished the race due to various reasons, the position and rank are set to `\N` in raw datasets. Finishing time is also set to `\N` in these cases, as well as when they finished with status like `+N laps`. This section modifies the data to standardize the columns datatype. A position/rank of `99` is set instead. And `NA` is used missing for time values.

In [11]:
merged_df['position'] = merged_df['position'].replace(r'\\N', 99, regex=True).astype(int)
merged_df['rank'] = merged_df['rank'].replace(r'\\N', 99, regex=True).astype(int)

merged_df['time'] = merged_df['time'].replace(r'\\N', 'NA', regex=True)

### Saving to CSV

In [12]:
# sorting values by - raceId, grid
merged_df = merged_df.sort_values(by=["raceId", "grid"])

# CSV output
merged_df.to_csv("../../data/positional_data.csv", index=False)